## Cell 1 — Install Required Libraries

In [1]:
# Install required libraries
!pip install transformers -q
!pip install scikit-learn -q

print("Libraries installed successfully.")

Libraries installed successfully.


## Cell 2 — Check GPU

In [2]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

if device.type == "cuda":
    print("GPU Name:", torch.cuda.get_device_name(0))
    print("GPU Memory:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB")
else:
    print("WARNING: No GPU detected. Go to Runtime > Change runtime type > T4 GPU")

Using device: cuda
GPU Name: Tesla T4
GPU Memory: 15.64 GB


## Cell 3 — Upload Your Files
Upload `edu_train.csv`, `edu_dev.csv`, and `label_encoder.pkl` when prompted.

In [6]:
from google.colab import files

print("Please upload: edu_train.csv, edu_dev.csv, label_encoder.pkl")
uploaded = files.upload()
print("\nFiles uploaded:", list(uploaded.keys()))

Please upload: edu_train.csv, edu_dev.csv, label_encoder.pkl


Saving label_encoder.pkl to label_encoder.pkl

Files uploaded: ['label_encoder.pkl']


## Cell 4 — Load and Preprocess Data

In [ ]:
import pandas as pd
import joblib
import numpy as np
from transformers import AutoTokenizer

# ── Load CSVs ──
train_df = pd.read_csv("edu_train_augmented.csv")
dev_df   = pd.read_csv("edu_dev.csv")

train_df = train_df[['source_article', 'updated_label']]
dev_df   = dev_df[['source_article', 'updated_label']]

# ── Load label encoder ──
label_encoder = joblib.load("label_encoder.pkl")

train_df['label_encoded'] = label_encoder.transform(train_df['updated_label'])
dev_df['label_encoded']   = label_encoder.transform(dev_df['updated_label'])

print("Label classes:", list(label_encoder.classes_))
print("\nTraining label distribution:")
print(train_df['updated_label'].value_counts())
print("\nTotal train samples:", len(train_df))
print("Total dev samples:", len(dev_df))

Label classes: ['ad hominem', 'ad populum', 'appeal to emotion', 'circular reasoning', 'equivocation', 'fallacy of credibility', 'fallacy of extension', 'fallacy of logic', 'fallacy of relevance', 'false causality', 'false dilemma', 'faulty generalization', 'intentional']

Training label distribution:
updated_label
faulty generalization     319
ad hominem                225
false causality           174
ad populum                158
circular reasoning        134
appeal to emotion         130
fallacy of logic          121
fallacy of relevance      114
intentional               112
false dilemma             110
fallacy of credibility    107
fallacy of extension      106
equivocation               39
Name: count, dtype: int64

Total train samples: 1849
Total dev samples: 300


/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.7.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


## Cell 5 — Tokenize Data

In [8]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

print("Tokenizing training data...")
train_encodings = tokenizer(
    train_df['source_article'].tolist(),
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

print("Tokenizing dev data...")
dev_encodings = tokenizer(
    dev_df['source_article'].tolist(),
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

train_labels = torch.tensor(train_df['label_encoded'].values)
dev_labels   = torch.tensor(dev_df['label_encoded'].values)

print("\nTrain encodings shape:", train_encodings['input_ids'].shape)
print("Dev encodings shape:", dev_encodings['input_ids'].shape)
print("Tokenization complete.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizing training data...
Tokenizing dev data...

Train encodings shape: torch.Size([1849, 128])
Dev encodings shape: torch.Size([300, 128])
Tokenization complete.


## Cell 6 — Prepare Dataset and Class Weights

In [9]:
from torch.utils.data import Dataset, DataLoader
from sklearn.utils.class_weight import compute_class_weight

# ── Dataset class ──
class FallacyDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = FallacyDataset(train_encodings, train_labels)
dev_dataset   = FallacyDataset(dev_encodings, dev_labels)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
dev_loader   = DataLoader(dev_dataset, batch_size=16)

# ── Compute class weights to fix imbalance ──
train_labels_numpy = train_df['label_encoded'].values

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels_numpy),
    y=train_labels_numpy
)

weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)

# ── Weighted loss function ──
loss_fn = torch.nn.CrossEntropyLoss(weight=weights_tensor)

print("Class weights computed:")
for i, (label, weight) in enumerate(zip(label_encoder.classes_, class_weights)):
    print(f"  {label}: {weight:.4f}")

Class weights computed:
  ad hominem: 0.6321
  ad populum: 0.9002
  appeal to emotion: 1.0941
  circular reasoning: 1.0614
  equivocation: 3.6469
  fallacy of credibility: 1.3293
  fallacy of extension: 1.3418
  fallacy of logic: 1.1755
  fallacy of relevance: 1.2476
  false causality: 0.8174
  false dilemma: 1.2930
  faulty generalization: 0.4459
  intentional: 1.2699


## Cell 7 — Load Model and Train

In [12]:
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import classification_report, f1_score

# ── Load model ──
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=13
)
model.to(device)

optimizer = AdamW(model.parameters(), lr=2e-5)

epochs = 4
best_f1 = 0

total_steps = len(train_loader) * epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=total_steps // 10,
    num_training_steps=total_steps
)

print("Starting training...\n")

for epoch in range(epochs):
    # ── Training ──
    model.train()
    total_train_loss = 0

    for batch in train_loader:
        optimizer.zero_grad()

        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # Use weighted loss instead of default outputs.loss
        loss = loss_fn(outputs.logits, labels)
        total_train_loss += loss.item()

        loss.backward()
        optimizer.step()
        scheduler.step()

    avg_train_loss = total_train_loss / len(train_loader)

    # ── Validation ──
    model.eval()
    total_val_loss = 0
    predictions  = []
    true_labels  = []

    with torch.no_grad():
        for batch in dev_loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            val_loss = loss_fn(outputs.logits, labels)
            total_val_loss += val_loss.item()

            preds = torch.argmax(outputs.logits, dim=1)
            predictions.extend(preds.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())

    avg_val_loss = total_val_loss / len(dev_loader)
    macro_f1     = f1_score(true_labels, predictions, average="macro")

    print(f"Epoch {epoch+1}/{epochs}")
    print(f"  Train Loss : {avg_train_loss:.4f}")
    print(f"  Val Loss   : {avg_val_loss:.4f}")
    print(f"  Macro F1   : {macro_f1:.4f}")

    if macro_f1 > best_f1:
        best_f1 = macro_f1
        torch.save(model.state_dict(), "bert_fallacy_model.pt")
        print("  ✅ Best model saved.\n")
    else:
        print()

print("=" * 40)
print(f"Training complete. Best Macro F1: {best_f1:.4f}")
print("=" * 40)
print("\nFinal Classification Report:")
print(classification_report(true_labels, predictions, target_names=label_encoder.classes_))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Starting training...

Epoch 1/4
  Train Loss : 2.5235
  Val Loss   : 2.3611
  Macro F1   : 0.2246
  ✅ Best model saved.

Epoch 2/4
  Train Loss : 2.1782
  Val Loss   : 1.9594
  Macro F1   : 0.4102
  ✅ Best model saved.

Epoch 3/4
  Train Loss : 1.7682
  Val Loss   : 1.7317
  Macro F1   : 0.4735
  ✅ Best model saved.

Epoch 4/4
  Train Loss : 1.5208
  Val Loss   : 1.6502
  Macro F1   : 0.5030
  ✅ Best model saved.

Training complete. Best Macro F1: 0.5030

Final Classification Report:
                        precision    recall  f1-score   support

            ad hominem       0.68      0.72      0.70        36
            ad populum       0.80      0.80      0.80        44
     appeal to emotion       0.37      0.50      0.42        14
    circular reasoning       0.52      0.78      0.62        18
          equivocation       0.00      0.00      0.00         5
fallacy of credibility       0.42      0.62      0.50         8
  fallacy of extension       0.21      0.36      0.26        1

## Cell 8 — Download Trained Model
This will download `bert_fallacy_model.pt` to your local machine.
Place it in your project at: `model/trained_model/bert_fallacy_model.pt`

In [13]:
from google.colab import files

files.download("bert_fallacy_model.pt")
print("Download started. Save the file to: model/trained_model/bert_fallacy_model.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download started. Save the file to: model/trained_model/bert_fallacy_model.pt
